# 01 · Extract features

**The single most consequential step in the project.**

The front end is 300M parameters.  Run it forward-only, once, and cache the
result; after that, training the head costs minutes per experiment rather than
a day.  Over a prep period that is the difference between five experiments and
several hundred, and it is why the whole project fits in a free GPU budget.

Roughly 30 minutes on a T4 for the ASVspoof training split.

**Two caches, not one.**  Codec degradation is applied *before* the front end,
so the baseline and codec-robust models need separate caches.  Conflating them
would silently compare a model against itself.

In [ ]:
# --- Colab setup -----------------------------------------------------------
# Run this first in every notebook.  Idempotent.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Keep the repo and all caches on Drive so a disconnect does not cost you
    # the feature extraction pass.
    PROJECT = Path("/content/drive/MyDrive/voice-integrity")
    if not PROJECT.exists():
        raise SystemExit(
            f"Upload or clone the repo to {PROJECT} first.\n"
            "  !git clone <your-repo-url> /content/drive/MyDrive/voice-integrity"
        )
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

# Repo-local model cache.  Set BEFORE importing transformers, or it will use
# the default location and the cache will not be portable to the demo machine.
os.environ["HF_HOME"] = str(PROJECT / "cache" / "huggingface")
os.environ["TORCH_HOME"] = str(PROJECT / "cache" / "torch")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("project:", PROJECT)
print("python :", sys.version.split()[0])

In [ ]:
if IN_COLAB:
    !pip install -q transformers speechbrain soundfile librosa pydantic pyyaml cryptography wandb
    !apt-get -qq install -y ffmpeg libopencore-amrnb-dev > /dev/null

# AMR-NB encoding is the one that silently goes missing.  If this prints
# nothing, your mobile-codec augmentation does nothing and the whole
# codec-robustness result quietly evaporates.
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -i amr || echo "AMR-NB ENCODER MISSING"

In [ ]:
import torch
print("cuda available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device         :", torch.cuda.get_device_name(0))
    print("memory         : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from vif.common.config import load_config
from vif.data.manifests import read_manifest

config = load_config("configs")
print("front end :", config.model.frontend.model_id)
print("window    :", config.model.audio.window_samples, "samples",
      f"({config.model.audio.window_seconds:.2f} s)")
print("frozen    :", config.model.frontend.frozen)

train_items = read_manifest("data/manifests/asvspoof19la_train.jsonl")
dev_items   = read_manifest("data/manifests/asvspoof19la_dev.jsonl")
print(f"\ntrain {len(train_items)}  dev {len(dev_items)}")

## Cache A — clean

Feeds the **baseline** model.  No augmentation of any kind.

In [ ]:
from vif.train.extract import extract_features, verify_cache

extract_features(
    train_items, config,
    out_dir="data/features/train_clean",
    device=DEVICE, augment=False, batch_size=8,
)
extract_features(
    dev_items, config,
    out_dir="data/features/dev_clean",
    device=DEVICE, augment=False, batch_size=8,
)

ok, detail = verify_cache("data/features/train_clean", len(train_items),
                          config.model.frontend.hidden_dim)
print(("ok  " if ok else "FAIL"), detail)

## Cache B — codec-degraded

Feeds the **codec-robust** model.  The augmenter is stochastic, so it writes
the realised condition back into the manifest — the per-codec breakdown at
evaluation time must reflect what actually happened, not what the config asked
for.

In [ ]:
from vif.data.augment import check_ffmpeg_codecs

status = check_ffmpeg_codecs(config.augment.codecs)
for name, available in status.items():
    print(f"  {name:<12} {'ok' if available else 'MISSING ENCODER'}")

if not any(status.values()):
    raise SystemExit("No codec is usable. The contribution of this project depends on this.")

In [ ]:
extract_features(
    train_items, config,
    out_dir="data/features/train_codec",
    device=DEVICE, augment=True, batch_size=8, seed=1234,
)
extract_features(
    dev_items, config,
    out_dir="data/features/dev_codec",
    device=DEVICE, augment=True, batch_size=8, seed=1234,
)
print("done")

In [ ]:
# What the augmenter actually applied.
from collections import Counter
from vif.data.manifests import read_manifest

realised = read_manifest("data/features/train_codec/manifest.jsonl")
for condition, count in Counter(i.condition for i in realised).most_common():
    print(f"  {condition:<24} {count:6d}")

## Cache size

Features are stored as float16.  Expect roughly 0.4 MB per utterance at
201 frames × 1024 dims, so a 25k-utterance split lands around 10 GB per cache.
Keep them on Drive, not on the Colab instance disk.

In [ ]:
import subprocess
for d in ["train_clean", "dev_clean", "train_codec", "dev_codec"]:
    path = Path("data/features") / d
    if path.exists():
        size = sum(f.stat().st_size for f in path.glob("*.npy")) / 1e9
        print(f"  {d:<14} {len(list(path.glob('*.npy'))):6d} files   {size:6.2f} GB")